<a href="https://colab.research.google.com/github/suremahitha2006-blip/speech-emotion-recognition/blob/main/notebooks/speech_emotion_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip "archive (1).zip"


Archive:  archive (1).zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of archive (1).zip or
        archive (1).zip.zip, and cannot find archive (1).zip.ZIP, period.


In [ ]:
!file "archive (1).zip"


archive (1).zip: Zip archive data, at least v4.5 to extract, compression method=deflate


In [ ]:
!apt-get install p7zip-full -y


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!7z x "archive (1).zip"



7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 66060288 bytes (63 MiB)

Extracting archive: archive (1).zip

ERRORS:
Headers Error
Unconfirmed start of archive


WARNINGS:
There are data after the end of archive

--
Path = archive (1).zip
Type = zip
ERRORS:
Headers Error
Unconfirmed start of archive
WARNINGS:
There are data after the end of archive
Physical Size = 14754289
Tail Size = 51305999

  0%     41% 40 - Actor_01/03-01-06-02-01-01-01.wav                                            81% 78 - Actor_02/03-01-03-02-02-01-02.wav                                           ERROR: Data Er

In [ ]:
import os
print(os.listdir())


['.config', 'archive (1).zip', 'Actor_02', 'Actor_01', 'sample_data']


In [ ]:
Use only the extracted actors (Actor_01 & Actor_02)

SyntaxError: invalid syntax (ipython-input-1100499119.py, line 1)

In [ ]:
!rm -rf Actor_*


In [ ]:
!7z x "archive (1).zip" -aos



7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 97517568 bytes (93 MiB)

Extracting archive: archive (1).zip

ERRORS:
Headers Error
Unconfirmed start of archive


WARNINGS:
There are data after the end of archive

--
Path = archive (1).zip
Type = zip
ERRORS:
Headers Error
Unconfirmed start of archive
WARNINGS:
There are data after the end of archive
Physical Size = 14754289
Tail Size = 82763279

  0%     58% 55 - Actor_01/03-01-08-01-02-02-01.wav                                           ERROR: Data Error : Actor_02/03-01-06-01-01-01-02.wav
100% 96 - Actor_02/03-01-06-01-01-01-02.wav                                           

In [ ]:
import os

actors = sorted([f for f in os.listdir() if f.startswith("Actor_")])
print("Total actors extracted:", len(actors))
print(actors)


Total actors extracted: 2
['Actor_01', 'Actor_02']


In [ ]:
import librosa

bad_files = []

for actor in actors:
    for file in os.listdir(actor):
        if file.endswith(".wav"):
            path = os.path.join(actor, file)
            try:
                librosa.load(path, duration=1)
            except:
                bad_files.append(path)

print("Corrupted files found:", len(bad_files))

# delete them
for bf in bad_files:
    os.remove(bf)

print("Corrupted files removed")


Corrupted files found: 0
Corrupted files removed


In [ ]:
total_files = 0
for actor in actors:
    total_files += len([f for f in os.listdir(actor) if f.endswith(".wav")])

print("Total usable audio files:", total_files)


Total usable audio files: 97


In [ ]:
from collections import Counter
import os

emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

emotion_counts = Counter()

for actor in ["Actor_01", "Actor_02"]:
    for file in os.listdir(actor):
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            emotion = emotion_map[emotion_code]
            emotion_counts[emotion] += 1

print("Emotion distribution:")
for emotion, count in emotion_counts.items():
    print(emotion, ":", count)


Emotion distribution:
calm : 16
angry : 16
fearful : 9
disgust : 8
surprised : 8
sad : 16
happy : 16
neutral : 8


In [ ]:
import numpy as np
import librosa
import os

X = []
y = []

def extract_mfcc(file_path):
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    return np.mean(mfcc.T, axis=0)

for actor in ["Actor_01", "Actor_02"]:
    for file in os.listdir(actor):
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            emotion = emotion_map[emotion_code]

            file_path = os.path.join(actor, file)
            mfcc = extract_mfcc(file_path)

            X.append(mfcc)
            y.append(emotion)

print("Features extracted:", len(X))


Features extracted: 97


In [ ]:
from sklearn.preprocessing import LabelEncoder

X = np.array(X)
y = np.array(y)

le = LabelEncoder()
y = le.fit_transform(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (97, 40)
y shape: (97,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

X = np.array(X)
y = np.array(y)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# One-hot encode
y_cat = to_categorical(y_encoded)

# CNN needs 3D input
X = X.reshape(X.shape[0], X.shape[1], 1)

print("X shape:", X.shape)
print("y shape:", y_cat.shape)


X shape: (97, 40, 1)
y shape: (97, 8)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y_encoded
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(40,1)),
    MaxPooling1D(2),
    Dropout(0.3),

    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(y_cat.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 38, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 19, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 19, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 17, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 8, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157,192 (614.03 KB)

 Trainable params: 157,192 (614.03 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test)
)


Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.0867 - loss: 12.6257 - val_accuracy: 0.1500 - val_loss: 3.7603
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.1995 - loss: 10.1446 - val_accuracy: 0.1500 - val_loss: 5.1491
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1387 - loss: 8.6939 - val_accuracy: 0.2000 - val_loss: 2.7823
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1284 - loss: 5.7050 - val_accuracy: 0.2000 - val_loss: 2.4444
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1110 - loss: 5.1945 - val_accuracy: 0.2000 - val_loss: 2.2197
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2862 - loss: 4.4030 - val_accuracy: 0.2500 - val_loss: 2.0319
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1726 - loss: 4.1170 - val_accuracy: 0.3000 - val_loss: 2.0186
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1977 - loss: 2.9348 - val_accuracy: 0.2000 - val_loss: 2.060

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step


In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
print("ROC-AUC:", roc_auc)


ROC-AUC: 0.5974264705882353
